# Loading Libraries

In [58]:

import os
import re
import pandas as pd
import unicodedata
from sklearn.model_selection import train_test_split
import sentencepiece as spm
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)


# Loading Dataset

In [59]:


base_dir = "/kaggle/input/urdu-dataset-20000"
file_name = "final_main_dataset.tsv"
data_path = os.path.join(base_dir, file_name)

df = pd.read_csv(data_path, sep="\t", encoding="utf-8")

print(" Dataset loaded successfully.")
print("Total samples:", len(df))
print("Columns:", list(df.columns))
print("\nSample rows:\n", df.head(3))


 Dataset loaded successfully.
Total samples: 20000
Columns: ['client_id', 'path', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment']

Sample rows:
                                            client_id  \
0  e53f84d151d6cc6d45a57decde08a99efe47d7751a4ca6...   
1  e53f84d151d6cc6d45a57decde08a99efe47d7751a4ca6...   
2  e53f84d151d6cc6d45a57decde08a99efe47d7751a4ca6...   

                           path                            sentence  up_votes  \
0  common_voice_ur_31771683.mp3  کبھی کبھار ہی خیالی پلاو بناتا ہوں         2   
1  common_voice_ur_31771684.mp3   اور پھر ممکن ہے کہ پاکستان بھی ہو         2   
2  common_voice_ur_31771685.mp3       یہ فیصلہ بھی گزشتہ دو سال میں         2   

   down_votes       age gender accents  variant locale  segment  
0           0  twenties   male     NaN      NaN     ur      NaN  
1           1  twenties   male     NaN      NaN     ur      NaN  
2           0  twenties   male     NaN      NaN     ur   

# 1. Preprocessing

### Data Cleaning and Normalization

In [60]:

def clean_urdu_text(text):
    if pd.isna(text):
        return ""
    
    # Normalize Unicode form
    import unicodedata
    text = unicodedata.normalize("NFC", text)

    # Replacement mappings for normalization
    replacements = {
        # Yeh / Kaf / Heh variants
        "ي": "ی", "ى": "ی", "ئ": "ی", "یٰ": "ی",
        "ك": "ک",
         "ھ": "ہ", "ۀ": "ہ", "ة": "ہ", "ۃ": "ہ",
        "ة": "ہ",  # taa marbuta → heh
        "ؤ": "و",  # hamza-on-waw → waw
        "أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا",
        "ۃ": "ہ",
        "ہٰ": "ہ",
        # Remove small vowel marks
        "ٖ": "", "ٗ": "", "ٞ": "", "ٓ": "",
    }
    
    for src, dest in replacements.items():
        text = text.replace(src, dest)
    
    # Remove diacritics (harakat), tatweel, and non-Urdu characters
    text = re.sub(r"[\u064B-\u0652\u0670\u0653-\u065F\u06D6-\u06ED]", "", text)
    text = re.sub(r"\u0640", "", text)  # tatweel (ـ)
    text = re.sub(r"[^\u0600-\u06FF\s]", "", text)  # keep only Urdu chars
    text = re.sub(r"\s+", " ", text).strip()  # normalize spaces
    
    return text

# Apply cleaning function
if 'sentence' in df.columns:
    df['clean_text'] = df['sentence'].apply(clean_urdu_text)
elif 'text' in df.columns:
    df['clean_text'] = df['text'].apply(clean_urdu_text)
else:
    df['clean_text'] = df.iloc[:, 0].apply(clean_urdu_text)

print("\nSample cleaned text:\n")
print(df['clean_text'].head())



Sample cleaned text:

0                   کبہی کبہار ہی خیالی پلاو بناتا ہوں
1                    اور پہر ممکن ہے کہ پاکستان بہی ہو
2                        یہ فیصلہ بہی گزشتہ دو سال میں
3                       ان کے بلے بازوں کے سامنے ہو گا
4    ابی جانور میں بطخ بگلا اور دوسرا ابی پرندہ شام...
Name: clean_text, dtype: object


### Handling  Missing or Empty Rows

In [61]:

# Count missing (NaN) values
missing_count = df['clean_text'].isna().sum()

# Count empty strings (after stripping whitespace)
empty_count = (df['clean_text'].str.strip() == "").sum()

print(f"\nMissing (NaN) rows in 'clean_text': {missing_count}")
print(f"Empty text rows in 'clean_text': {empty_count}")
print(f"Total problematic rows: {missing_count + empty_count}")




df = df[df['clean_text'].str.strip() != ""]
df = df.dropna(subset=['clean_text']).reset_index(drop=True)
print("\nCleaned dataset size:", len(df))


Missing (NaN) rows in 'clean_text': 0
Empty text rows in 'clean_text': 2
Total problematic rows: 2

Cleaned dataset size: 19998


### Tokenizaztion and Vocabulary Building

In [62]:
train_file = "urdu_chat_data.txt"
df["clean_text"].to_csv(train_file, index=False, header=False, sep="\n", encoding="utf-8-sig", quoting=3)

print("Existing urdu dataset loaded and saved to urdu_chat_data.txt")


# Train SentencePiece Model (BPE)

model_prefix = "urdu_bpe"
vocab_size = 8000

spm.SentencePieceTrainer.train(
    input=train_file,
    model_prefix=model_prefix,
    vocab_size=vocab_size,
    model_type="bpe",
    character_coverage=1.0,
    user_defined_symbols=["آ", "أ", "إ", "ء", "ئ", "ؤ", "ۃ", "ہ", "ھ", "یٰ"],
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3,
    byte_fallback=False
)

print("\n SentencePiece model trained successfully with Urdu-specific symbols!")


#  Load and Test Tokenizer

sp = spm.SentencePieceProcessor()
sp.load(f"{model_prefix}.model")


# Test on Urdu Sentences

test_sentences = [
    "میں آج بہت خوش ہوں",
    "پاکستان ایک خوبصورت ملک ہے",
    "ہمیں اپنی زبان پر فخر ہونا چاہیے",
    "تعلیم کامیابی کی کنجی ہے",
    "کل موسم بہت سرد تھا",
    "میں کتابیں پڑھنا پسند کرتا ہوں"
]

print("\n=== Testing Urdu BPE Tokenization with Custom Urdu Characters ===\n")

for i, sentence in enumerate(test_sentences, 1):
    encoded_pieces = sp.encode(sentence, out_type=str)
    encoded_ids = sp.encode(sentence, out_type=int)
    decoded_text = sp.decode(encoded_ids)

    print(f"Sentence {i}: {sentence}")
    print(f"Subwords   : {encoded_pieces}")
    print(f"Encoded IDs: {encoded_ids}")
    print(f"Decoded    : {decoded_text}\n")

print(" Urdu BPE Tokenizer test completed successfully.")

Existing urdu dataset loaded and saved to urdu_chat_data.txt

 SentencePiece model trained successfully with Urdu-specific symbols!

=== Testing Urdu BPE Tokenization with Custom Urdu Characters ===

Sentence 1: میں آج بہت خوش ہوں
Subwords   : ['▁میں', '▁', 'آ', 'ج', '▁ب', 'ہ', 'ت', '▁خوش', '▁', 'ہ', 'وں']
Encoded IDs: [30, 7953, 4, 7969, 18, 11, 7962, 385, 7953, 11, 34]
Decoded    : میں آج بہت خوش ہوں

Sentence 2: پاکستان ایک خوبصورت ملک ہے
Subwords   : ['▁پاکستان', '▁ایک', '▁خوبصورت', '▁ملک', '▁', 'ہ', 'ے']
Encoded IDs: [97, 79, 2060, 220, 7953, 11, 7960]
Decoded    : پاکستان ایک خوبصورت ملک ہے

Sentence 3: ہمیں اپنی زبان پر فخر ہونا چاہیے
Subwords   : ['▁', 'ہ', 'میں', '▁اپنی', '▁زبان', '▁پر', '▁فخر', '▁', 'ہ', 'ونا', '▁چا', 'ہ', 'یے']
Encoded IDs: [7953, 11, 372, 225, 833, 62, 3460, 7953, 11, 223, 141, 11, 60]
Decoded    : ہمیں اپنی زبان پر فخر ہونا چاہیے

Sentence 4: تعلیم کامیابی کی کنجی ہے
Subwords   : ['▁تعلیم', '▁کامیابی', '▁کی', '▁ک', 'نجی', '▁', 'ہ', 'ے']
Encoded IDs: [910, 

In [66]:
# # ========================================
# # Step 4: Train/Validation/Test Split
# # ========================================

# from sklearn.model_selection import train_test_split

# # If labels exist, split text and labels together
# if 'label' in df.columns:
#     # Step 1: Split into Train (80%) and Temp (20%)
#     X_train, X_temp, y_train, y_temp = train_test_split(
#         df['clean_text'], df['label'], test_size=0.2, random_state=42
#     )

#     # Step 2: Split Temp (20%) into Validation (10%) and Test (10%)
#     X_val, X_test, y_val, y_test = train_test_split(
#         X_temp, y_temp, test_size=0.5, random_state=42
#     )
# else:
#     # Step 1: Split into Train (80%) and Temp (20%)
#     X_train, X_temp = train_test_split(
#         df['clean_text'], test_size=0.2, random_state=42
#     )

#     # Step 2: Split Temp (20%) into Validation (10%) and Test (10%)
#     X_val, X_test = train_test_split(
#         X_temp, test_size=0.5, random_state=42
#     )

#     y_train = y_val = y_test = None

# # Print summary
# print("\nTrain/Validation/Test Split Completed:")
# print("Training samples:", len(X_train))
# print("Validation samples:", len(X_val))
# print("Testing samples:", len(X_test))



pairs = []
for client in df['client_id'].unique():
    client_sentences = df[df['client_id'] == client]['sentence'].tolist()
    for i in range(len(client_sentences)-1):
        pairs.append((client_sentences[i], client_sentences[i+1]))

input_texts, target_texts = zip(*pairs)

# Split into train/val/test
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(input_texts, target_texts, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Print first 5 training samples
print("=== Sample Train Data ===\n")
for i in range(5):
    print(f"Input  : {X_train[i]}")
    print(f"Target : {y_train[i]}")
    print("-" * 50)

# Print first 5 validation samples
print("\n=== Sample Validation Data ===\n")
for i in range(5):
    print(f"Input  : {X_val[i]}")
    print(f"Target : {y_val[i]}")
    print("-" * 50)

# Print first 5 test samples
print("\n=== Sample Test Data ===\n")
for i in range(5):
    print(f"Input  : {X_test[i]}")
    print(f"Target : {y_test[i]}")
    print("-" * 50)



=== Sample Train Data ===

Input  : نہ گھڑی جاتیں۔
Target : اس صورتِ حال میں ہمارے نزدیک
--------------------------------------------------
Input  : چیزوں کا تصور کرنے میں مشکل پیش آتی ہے
Target : لیکن کچھ ذکر چکوال کا بھی ہو جائے۔
--------------------------------------------------
Input  : مزدور کا عالمی دن یا لیبر ڈے
Target : کا مکمل کنٹرول ہے۔
--------------------------------------------------
Input  : عرفان خان اور سونالی بیندر ے
Target : انقلاب توچند ہزارافراد اور چندسو گز تک محدود تھا۔
--------------------------------------------------
Input  : بھارت میں ہونے والا ورلڈ کپ کبڈی ٹورنامنٹ منسوخ
Target : کمرے میں تمام چیزیں بکھری پڑی تھیں
--------------------------------------------------

=== Sample Validation Data ===

Input  : مگر ان بیٹسمینوں پر تنقید کرنے کے بجائے
Target : ایک بہادر جرنیل دوسرے کی قدر جانتا ہے
--------------------------------------------------
Input  : صارفین نے کنکشن منقطع ہونے کے باوجودغیرقانونی بجلی استعمال کی
Target : چوہدری نثار کو بھی داد دینی چاہیے۔
-----

In [44]:

# Save Cleaned Data

cleaned_path = "/kaggle/working/cleaned_urdu_dataset.csv"
df.to_csv(cleaned_path, index=False, encoding='utf-8-sig')
print(f"\nCleaned dataset saved to: {cleaned_path}")



Cleaned dataset saved to: /kaggle/working/cleaned_urdu_dataset.csv


In [45]:
# ========================================
# Step 6: Sanity Check
# ========================================

print("\nRandom Samples from Cleaned Text:\n")
print(df['clean_text'].sample(5, random_state=1).values)


Random Samples from Cleaned Text:

['اس کی سب سے بڑی وجہ اس کی مسلسل کم ہوتی قیمت تہی' 'ہمیشہ باییں طرف چلیے'
 'یو کے' 'تو چہوٹی عصبیتیں باہم دست وگریباں ہونے لگتی ہیں۔'
 'دینے لگا ہے بوسہ بغیر التجا کیے']


# 2. Model Architecture

In [46]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

### Positional Encoding

In [47]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        # Create a positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                             (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # Shape: (1, max_len, d_model)
        
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x


### Multi-Head Attention

In [48]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Q, K, V shape: (batch, heads, seq_len, d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        output = torch.matmul(attn, V)
        return output, attn

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)

        # Linear projections
        Q = self.q_linear(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
        K = self.k_linear(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)
        V = self.v_linear(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1,2)

        # Apply attention
        out, attn = self.scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1,2).contiguous().view(batch_size, -1, self.d_model)
        return self.out(out)


### Feed Forward Network

In [49]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(FeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))


### Encoder Decoder Layer

In [52]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Multi-head attention + residual
        attn_out = self.mha(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Feed-forward + residual
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x




class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # Self-attention
        self_attn_out = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_out))

        # Encoder-Decoder attention
        enc_dec_out = self.enc_dec_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(enc_dec_out))

        # Feed-forward
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_out))
        return x


### Full Encoder Decoder layer

In [53]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout=0.1):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, mask)
        return x



class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout=0.1):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        logits = self.fc_out(x)
        return logits


### Complete Transformer

In [54]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, num_layers=2, 
                 num_heads=2, d_ff=512, max_len=100, dropout=0.1):
        super(Transformer, self).__init__()
        self.encoder = Encoder(src_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_layers, num_heads, d_ff, max_len, dropout)

    def make_tgt_mask(self, tgt):
        # Prevent decoder from attending to future tokens
        size = tgt.size(1)
        mask = torch.tril(torch.ones(size, size)).unsqueeze(0)  # (1, seq_len, seq_len)
        return mask.to(tgt.device)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_output = self.encoder(src, src_mask)
        if tgt_mask is None:
            tgt_mask = self.make_tgt_mask(tgt)
        out = self.decoder(tgt, enc_output, src_mask, tgt_mask)
        return out


In [55]:
d_model = 256          # Embedding dimensions
num_heads = 2          # Multi-head attention heads
num_layers = 2         # Encoder & Decoder layers
d_ff = 512             # Feed-forward network hidden size
dropout = 0.1          # Dropout rate
batch_size = 32
learning_rate = 1e-4
num_epochs = 20
max_len = 100          # Maximum sequence length



bos_id = 2
eos_id = 3
pad_id = 0



from torch.utils.data import Dataset, DataLoader

class UrduChatDataset(Dataset):
    def __init__(self, input_texts, target_texts, tokenizer, max_len=100):
        self.input_texts = input_texts
        self.target_texts = target_texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.input_texts)

    def __getitem__(self, idx):
        src_ids = [bos_id] + self.tokenizer.encode(self.input_texts[idx], out_type=int) + [eos_id]
        tgt_ids = [bos_id] + self.tokenizer.encode(self.target_texts[idx], out_type=int) + [eos_id]

        # Pad sequences
        src_ids = src_ids[:self.max_len] + [pad_id]*(self.max_len - len(src_ids))
        tgt_ids = tgt_ids[:self.max_len] + [pad_id]*(self.max_len - len(tgt_ids))

        return torch.tensor(src_ids), torch.tensor(tgt_ids)



train_dataset = UrduChatDataset(X_train, y_train, sp, max_len=max_len)
val_dataset = UrduChatDataset(X_val, y_val, sp, max_len=max_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)



import torch.optim as optim

model = Transformer(src_vocab_size=vocab_size,
                    tgt_vocab_size=vocab_size,
                    d_model=d_model,
                    num_layers=num_layers,
                    num_heads=num_heads,
                    d_ff=d_ff,
                    max_len=max_len,
                    dropout=dropout)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)



for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        
        optimizer.zero_grad()
        
        # Decoder input: all tokens except last
        decoder_input = tgt[:, :-1]
        # Decoder target: all tokens except first
        decoder_target = tgt[:, 1:]
        
        output = model(src, decoder_input)  # shape: (batch, seq_len, vocab_size)
        output = output.reshape(-1, vocab_size)
        decoder_target = decoder_target.reshape(-1)
        
        loss = criterion(output, decoder_target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")
    
    # Optional: validate BLEU score on val_loader here



torch.save(model.state_dict(), "best_model.pt")


TypeError: 'NoneType' object is not subscriptable